# Train Dewy's "Hey Dewy" wake word

This trains a tiny model that runs on Dewy's ESP32-S3 and listens for **"Hey Dewy"**, using
[microWakeWord](https://github.com/kahrendt/microWakeWord) (Apache 2.0). It's adapted from microWakeWord's
`basic_training_notebook.ipynb`.

**Before you start** (works in Colab, or on your own NVIDIA PC: see README.md and run `setup_wsl.sh` first)
1. In Colab: **Runtime > Change runtime type > T4 GPU**. On your own PC, skip this and the install cell.
2. Run the cells from top to bottom. The first cell installs things; **restart the session** when it finishes
   (Runtime > Restart session), then carry on from the second cell.
3. It takes a while: generating voices and downloading background sounds can take 30-60 minutes, training another
   1-2 hours on Colab's free GPU.

**What it does**
- Makes thousands of computer-generated voices saying "Hey Dewy" (in a few spellings, so the voices pronounce it
  different ways).
- Makes look-alike phrases that must *not* wake Dewy ("hey buddy", "do we", "hey Judy"...).
- Mixes everything with background noise, music and room echo, then trains.
- At the end you download `hey_dewy_model.h` (the model for the firmware) and `hey_dewy.json` (its settings).

**Expect to experiment.** microWakeWord's author says a first model often isn't good enough for daily use: it may
be hard to trigger or wake up by mistake. The cells marked **TRY** are the ones to change between rounds.

The background-sound datasets have mixed licences, so the model is for personal, non-commercial use.


In [ ]:
# Installs microWakeWord. In Colab, restart the session after this is finished.
# On your own PC, setup_wsl.sh has already installed everything, so this does nothing.
import importlib.util, platform

if importlib.util.find_spec("microwakeword") is not None:
    print("microWakeWord is already installed.")
else:
    if platform.system() == "Darwin":
        # `pymicro-features` is installed from a fork to support building on macOS
        !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

    # `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
    !pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

    !git clone https://github.com/kahrendt/microWakeWord
    !pip install -e ./microWakeWord

In [ ]:
# Settings for this notebook. TRY: add spellings that sound like how she says it.
# Phonetic spellings help the computer voices say it right; listen to the sample below.
WAKE_SPELLINGS = ["hey dewy", "hey doo ee", "hey dooey"]
SAMPLES_PER_SPELLING = 2500      # TRY: more samples usually help (5000+)

# Phrases that sound close and must NOT wake Dewy. TRY: add any that trigger it by mistake.
NEGATIVE_PHRASES = ["hey buddy", "hey dude", "hey judy", "hey julie", "hey you", "do we", "dewey",
                    "hey doctor", "hey drew", "duty", "hey dewey decimal", "hey siri", "okay google",
                    "hey there", "hey, do it", "how do we"]
SAMPLES_PER_NEGATIVE = 300


# Which Python runs the voice generator. On your own PC it lives in its own environment (setup_wsl.sh sets this).
import os, sys
PIPER_PY = os.environ.get("DEWY_PIPER_PYTHON", sys.executable)


Listen to one generated "Hey Dewy". If it sounds wrong, change `WAKE_SPELLINGS` above.

In [ ]:
# Generates 1 sample of the target word for manual verification.

target_word = WAKE_SPELLINGS[0]

import os
import sys
import platform

from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    if platform.system() == "Darwin":
        !git clone -b mps-support https://github.com/kahrendt/piper-sample-generator
    else:
        # v2.0.0 is pinned: newer versions of this repo no longer have generate_samples.py
        !git clone -b v2.0.0 https://github.com/rhasspy/piper-sample-generator

    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

    # Install system dependencies
    !pip install torch torchaudio piper-phonemize-cross==1.2.1

    if "piper-sample-generator/" not in sys.path:
        sys.path.append("piper-sample-generator/")

!{PIPER_PY} piper-sample-generator/generate_samples.py "{target_word}" \
--max-samples 1 \
--batch-size 1 \
--output-dir sample_check

Audio("sample_check/0.wav", autoplay=True)

Generate the wake word in every spelling, then the look-alike phrases. Files are renamed so nothing is overwritten.

In [ ]:
import os, shutil, glob, subprocess, sys

def generate(text, count, out_dir):
    tmp = "tmp_gen"
    shutil.rmtree(tmp, ignore_errors=True)
    subprocess.run([PIPER_PY, "piper-sample-generator/generate_samples.py", text, "--max-samples", str(count),
                    "--batch-size", "100", "--output-dir", tmp], check=True)
    os.makedirs(out_dir, exist_ok=True)
    tag = text.replace(" ", "_").replace(",", "")
    for i, f in enumerate(sorted(glob.glob(tmp + "/*.wav"))):
        shutil.move(f, os.path.join(out_dir, f"{tag}_{i}.wav"))

if not os.path.exists("generated_samples"):
    for spelling in WAKE_SPELLINGS:
        generate(spelling, SAMPLES_PER_SPELLING, "generated_samples")
if not os.path.exists("negative_samples"):
    for phrase in NEGATIVE_PHRASES:
        generate(phrase, SAMPLES_PER_NEGATIVE, "negative_samples")

print(len(glob.glob("generated_samples/*.wav")), "wake word clips,", len(glob.glob("negative_samples/*.wav")), "look-alike clips")


Download room echoes, background noise and music for mixing in. This is the slow part.

In [ ]:
# Downloads audio data for augmentation. This can be slow!
# Borrowed from openWakeWord's automatic_model_training.ipynb, accessed March 4, 2024
#
# **Important note!** The data downloaded here has a mixture of difference
# licenses and usage restrictions. As such, any custom models trained with this
# data should be considered as appropriate for **non-commercial** personal use only.


import datasets
import scipy
import os

import numpy as np

from pathlib import Path
from tqdm import tqdm

## Download MIR RIR data

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
    !wget -O {out_dir} {link}
    !cd audioset && tar -xf bal_train09.tar

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset
# https://github.com/mdeff/fma
# (Third-party mchl914 extra small set)

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_dir = f"fma/{fname}"
    !wget -O {out_dir} {link}
    !cd {output_dir} && unzip -q {fname}

    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    fma_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(fma_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))


Set up the mixing. **TRY**: the noise levels (`background_min_snr_db`, `background_max_snr_db`).

In [ ]:
# Sets up the augmentations for the wake word clips.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )


In [ ]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

Turn the wake word clips into training features.

In [ ]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

Do the same for the look-alike phrases (these teach the model what *not* to react to).

In [ ]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

clips = Clips(input_directory='negative_samples', file_pattern='*.wav', max_clip_duration_s=None,
              remove_silence=False, random_split_seed=10, split_count=0.1)

output_dir = 'negative_phrase_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'lookalike_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

Download microWakeWord's ready-made negative sets (ordinary speech, a dinner party, silence and noise).

In [ ]:
# Downloads pre-generated spectrogram features (made for microWakeWord in
# particular) for various negative datasets. This can be slow!

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname

        zip_path = f"negative_datasets/{fname}"
        !wget -O {zip_path} {link}
        !unzip -q {zip_path} -d {output_dir}

Training settings. **TRY**: `negative_class_weight` (higher = fewer false wake-ups, harder to trigger) and `training_steps`.

In [ ]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = (
    "trained_models/hey_dewy"
)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {   # look-alike phrases made above. TRY: raise penalty_weight if it wakes on them
        "features_dir": "negative_phrase_features",
        "sampling_weight": 8.0,
        "penalty_weight": 2.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [20000]   # TRY: more steps if it is hard to trigger

# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

config["learning_rates"] = [
    0.001,
]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (
    500  # Test the validation sets after every this many steps
)
config["clip_duration_ms"] = (
    1500  # Maximum length of wake word that the streaming model will accept
)

# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(os.path.join("training_parameters.yaml"), "w") as file:
    documents = yaml.dump(config, file)

Train. On Colab this can look stuck for minutes at a time; it isn't. On your own GPU it prints progress. At the end it prints test results, including false accepts per hour: aim for well under 1.

In [ ]:
# Trains a model. When finished, it will quantize and convert the model to a
# streaming version suitable for on-device detection.
# It will resume if stopped, but it will start over at the configured training
# steps in the yaml file.
# Change --train 0 to only convert and test the best-weighted model.
# On Google colab, it doesn't print the mini-batch results, so it may appear
# stuck for several minutes! Additionally, it is very slow compared to training
# on a local GPU.

!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

Package the model for Dewy. `PROBABILITY_CUTOFF` is how sure the model must be: look at the test results above,
and pick a cutoff where recall is still high but false accepts per hour are low. `SLIDING_WINDOW` is how many
inferences in a row (30 ms each) are averaged.

In [ ]:
import json, os
try:
    from google.colab import files   # only exists in Colab
except ImportError:
    files = None

PROBABILITY_CUTOFF = 0.97   # TRY: lower = easier to wake, more false wake-ups
SLIDING_WINDOW = 5
TENSOR_ARENA = 30000        # bytes of working memory on the ESP32; raise it if the model won't load

model_path = "trained_models/hey_dewy/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
data = open(model_path, "rb").read()

# The model as a C array, ready to drop into firmware/dewy/
lines = ["// Hey Dewy wake word model, trained with microWakeWord. Generated by voice/wakeword/train_hey_dewy.ipynb.",
         "#pragma once", "#include <stdint.h>",
         f"#define HEY_DEWY_PROBABILITY_CUTOFF {PROBABILITY_CUTOFF}f",
         f"#define HEY_DEWY_SLIDING_WINDOW {SLIDING_WINDOW}",
         f"#define HEY_DEWY_TENSOR_ARENA {TENSOR_ARENA}",
         f"static const unsigned int HEY_DEWY_MODEL_LEN = {len(data)};",
         "alignas(16) static const uint8_t HEY_DEWY_MODEL[] = {"]
for i in range(0, len(data), 16):
    lines.append("  " + ", ".join(f"0x{b:02x}" for b in data[i:i + 16]) + ",")
lines.append("};")
open("hey_dewy_model.h", "w").write("\n".join(lines) + "\n")

json.dump({"wake_word": "Hey Dewy", "probability_cutoff": PROBABILITY_CUTOFF, "sliding_window_size": SLIDING_WINDOW,
           "feature_step_size": 10, "tensor_arena_size": TENSOR_ARENA, "model_bytes": len(data),
           "spellings": WAKE_SPELLINGS, "negative_phrases": NEGATIVE_PHRASES}, open("hey_dewy.json", "w"), indent=2)
print(f"model: {len(data)} bytes")
if files:
    files.download("hey_dewy_model.h")
    files.download("hey_dewy.json")
    files.download(model_path)
else:
    import shutil
    shutil.copy(model_path, "stream_state_internal_quant.tflite")
    print("Saved in", os.getcwd(), ": hey_dewy_model.h, hey_dewy.json, stream_state_internal_quant.tflite")
